# Advanced MCP in Python

We dive deeper into implementing MCP servers using the [fastmcp](https://gofastmcp.com/getting-started/welcome) library. Specifically, we will learn about:

- [Resources](https://gofastmcp.com/servers/resources)
- [Tools](https://gofastmcp.com/servers/tools)
- [Prompts](https://gofastmcp.com/servers/prompts)

In [1]:
# pip install mcp fastmcp
import sys, os
import pandas as pd
import sqlite3, json
from fastmcp import FastMCP

In [2]:
mcp = FastMCP("Resilient Cities Business Server")
database = r'C:\Users\hanisaf\Downloads\resilient_cities.db'
conn = sqlite3.connect(database)

In [3]:
df1 = pd.read_sql('select * from city_customers', conn)

In [4]:
cities = df1.city.to_list()

# Resources

In [5]:
@mcp.resource("sustainability://athens")
def get_athens_sustainability() -> str:
    """Provides sustainability text for Athens, Greece"""
    conn = sqlite3.connect(database)
    res=conn.execute("select sustainability from city_sustainability where city='Athens'")
    row = res.fetchone()
    return row[0] if row else ""

In [6]:
for c in cities:
    def make_resource(city):
        uri =  f"sustainability://{city.replace(' ', '_')}"
        name = f"{city.replace(' ', '_')}"
        description = f"""Provides sustainability text for {city}"""
        @mcp.resource(
            uri=uri,
            name=name,                    # resource name shown to clients
            description=description
        )
        def get_city_sustainability() -> str:
            
            conn = sqlite3.connect(database)
            res=conn.execute("select sustainability from city_sustainability where city=?", (city,))
            row = res.fetchone()
            return row[0] if row else ""
    make_resource(c)

In [7]:
@mcp.tool
def query_db(query : str) -> str:
    """Execute query against the database. The schema of the database is below:

    -- Table: city_customers (city TEXT, country TEXT, total_customers INTEGER, volume_esg INTEGER, volume_other INTEGER)
    -- Table: city_sustainability (city TEXT, country TEXT, sustainability TEXT)
    
    It returns a pandas dataframe dumped as json text"""
    conn = sqlite3.connect(database)
    df = pd.read_sql(query, conn)
    return json.dumps(df.to_json())

In [8]:
@mcp.prompt
def lookup_metric(metric: str) -> str:
    """List the top three cities according to a metric"""
    return f"Query the database and lookup the top three cities according to '{metric}'"

In [9]:
# test if the script is running as a standalone program or a notebook
if __name__ == '__main__' and '__file__' in globals():
    print("Running as a script")
    interactive = False
    # additional testing here ...
    # print(get_athens_sustainability())
    mcp.run()
else:
    print("Running as a notebook")
    interactive = True

Running as a notebook


In [10]:
if not interactive:
    sys.exit(1)

In [11]:
print(get_athens_sustainability()[:1000])

Executive Summary
The City of Athens, a venerable yet intricate city of
near 700.000 residents, part of a 3.75 million people
metropolis, is for several years now facing a serious
socio-economic crisis. Through concerted efforts, the
city has managed to survive, to adapt and to transform
into a more creative and collaborative city. The struggle
is by no means over. This last year, calling on the insights and expertise of hundreds of stakeholders, from opinion leaders and academics to women migrants and the
homeless, the city drafted its Resilience Strategy. This is
a set of practicable actions which first of all strengthen
and scales up what has made our city stronger: formal
and informal networks and alliances. Athens Resilience
Strategy offers a set of new integrated ways to prepare
and protect our most vulnerable from future shocks and
stresses that the city will face.
Boosting the city's resilience means creating new as well as revitalizing existing open and green public spaces. Th

In [12]:
print(query_db("select distinct city from city_customers"))

"{\"city\":{\"0\":\"Amman\",\"1\":\"Athens\",\"2\":\"Bangkok\",\"3\":\"Berkeley\",\"4\":\"Boulder\",\"5\":\"Bristol\",\"6\":\"Byblos\",\"7\":\"Dakar\",\"8\":\"El Paso\",\"9\":\"Glasgow\",\"10\":\"Greater Christchurch\",\"11\":\"Medelin\",\"12\":\"Melbourne\",\"13\":\"Mexico City\",\"14\":\"New Orleans\",\"15\":\"Norfolk\",\"16\":\"Oakland\",\"17\":\"Pittsburgh\",\"18\":\"Quito\",\"19\":\"Ramallah\",\"20\":\"Rio de Janeiro\",\"21\":\"Rotterdam\",\"22\":\"San Francisco\",\"23\":\"Santa Fe\",\"24\":\"Semarang\",\"25\":\"Surat\",\"26\":\"Thessaloniki\",\"27\":\"Toyama\",\"28\":\"Vejles\",\"29\":\"Wellington\"}}"


In [13]:
print(lookup_metric("total_customers"))

Query the database and lookup the top three cities according to 'total_customers'
